In [1]:
import os
from dotenv import load_dotenv
from pathlib import Path
# Load variables from .env into environment
load_dotenv()

# Read the key
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
MODEL_NAME = os.getenv("MODEL_NAME")
VISION_MODEL_NAME = os.getenv("VISION_MODEL_NAME")

In [2]:
current_dir = Path.cwd()
BASE_DIR = current_dir.parent.parent
root_folder = BASE_DIR / "dataset/"
# List only directories inside subfolder
sub_folders = [f for f in root_folder.iterdir() if f.is_dir()]

doc_dict = {}

for folder in sub_folders:
    parent_folder = BASE_DIR / folder.name
    subfolder = parent_folder.joinpath(folder)
    iterator = list(subfolder.iterdir())
    first_two = iterator[0:1]   # TypeError!
    ext = os.path.splitext(first_two[0])[1][1:]
    if ext.lower() in ['jpg', 'jpeg', 'png', 'tiff', 'bmp', 'gif']: 
        # doc_dict[folder.name] = extract_text_from_image(str(first_two[0]))
        doc_dict[folder.name] = str(first_two[0])

In [10]:
doc_dict

{'claim_forms': 'd:\\Project\\AI\\Project\\dataset\\claim_forms\\claim_PT_15075.png',
 'discharge_summaries': 'd:\\Project\\AI\\Project\\dataset\\discharge_summaries\\discharge_PT_15075.png',
 'id_documents': 'd:\\Project\\AI\\Project\\dataset\\id_documents\\id_PT_15075.png',
 'policy_amendments': 'd:\\Project\\AI\\Project\\dataset\\policy_amendments\\amend_PT_15075.png',
 'prescriptions': 'd:\\Project\\AI\\Project\\dataset\\prescriptions\\rx_PT_15075.png',
 'unknown': 'd:\\Project\\AI\\Project\\dataset\\unknown\\unknown_bank_statement_001.png'}

In [ ]:
import base64
import os
import mimetypes
import json
from pathlib import Path
from typing import Union, Optional, Dict, Any, List
from groq import Groq

# Initialize once — reuses the connection
_client: Optional[Groq] = None

def _get_client() -> Groq:
    global _client
    if _client is None:
        api_key = os.environ.get("GROQ_API_KEY")
        if not api_key:
            raise ValueError("GROQ_API_KEY environment variable not set.")
        _client = Groq(api_key=api_key)
    return _client


def _encode_image(image_path: str) -> tuple[str, str]:
    """Returns (base64_string, mime_type)."""
    path = Path(image_path)
    if not path.is_file():
        raise FileNotFoundError(f"Image not found: {image_path}")

    mime_type, _ = mimetypes.guess_type(path.name)
    mime_type = mime_type or "image/jpeg"

    with open(path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")
    return encoded, mime_type


def read_image(image: Union[str, List[str]],prompt: str = "Extract all text and key data from this image.", model: str = "qwen/qwen3.8-27b",*,
    json_schema: Optional[Dict[str, Any]] = None,temperature: float = 1.0,max_tokens: int = 2048,stream: bool = False,) -> Union[str, Dict[str, Any]]:
    """
    Read and extract data from one or more images using Groq's vision models.

    Args:
        image:       Local file path, URL, or a list of either (max 3).
        prompt:      Instruction for the model.
        model:       Groq vision model ID.
        json_schema: Optional JSON schema — if provided, returns a dict.
        temperature: Sampling temperature.
        max_tokens:  Max completion tokens.
        stream:      Stream the response (returns generator).

    Returns:
        String, dict (if json_schema given), or generator (if stream=True).
    """
    client = _get_client()

    # Normalize to list
    images = [image] if isinstance(image, str) else image
    if not images:
        raise ValueError("At least one image is required.")
    if len(images) > 3:
        raise ValueError("Groq supports a maximum of 3 images per request.")

    # Build content blocks
    content: List[Dict[str, Any]] = [{"type": "text", "text": prompt}]

    for img in images:
        if img.startswith(("http://", "https://")):
            url = img
        else:
            b64, mime = _encode_image(img)
            url = f"data:{mime};base64,{b64}"
        content.append({
            "type": "image_url",
            "image_url": {"url": url},
        })

    # Build request kwargs
    kwargs: Dict[str, Any] = {
        "model": model,
        "messages": [{"role": "user", "content": content}],
        "temperature": temperature,
        "max_completion_tokens": max_tokens,
        "stream": stream,
    }

    if json_schema:
        kwargs["response_format"] = {"type": "json_object"}
        # Inject schema into the prompt so the model knows what to produce
        schema_hint = (
            "\n\nRespond ONLY with valid JSON matching this schema:\n"
            f"{json.dumps(json_schema, indent=2)}"
        )
        content[0]["text"] += schema_hint

    completion = client.chat.completions.create(**kwargs)

    if stream:
        return _stream_response(completion)

    raw = completion.choices[0].message.content

    if json_schema:
        try:
            return json.loads(raw)
        except json.JSONDecodeError as e:
            raise ValueError(f"Model did not return valid JSON: {raw}") from e

    return raw


def _stream_response(completion):
    """Yield text chunks from a streaming completion."""
    for chunk in completion:
        delta = chunk.choices[0].delta.content
        if delta:
            yield delta

In [9]:
kyc_raw = read_image(doc_dict['claim_forms'])
print(kyc_raw)

Here is the text extraction and key data, paraphrased as requested:

**Header:** A patient is expected from New York's Hudson medical providers at six local ophthalmoses with many, following a company clinic from the week. He reports experiencing sudden head concerns within $3,\perp$, to the right and feeling of poor orientation alongside his headache concern (some days). One period lasting, his visual assessment showed irregular color features; CT confirmation due only; confirmed all indications were present or near confirmation areas. A thorough exam supports at least small diagnostic and physical labs which have resolved but yet are undulations — one hospital decision for attentional control within a second’s second moment was able prior? The prognosis/secondary procedure may require minor more symptoms without clear time—confirm any important to all recent doctor orders including new labs, X-LIF-TOTAL-LEFT INTRANE CASH-CHECK RECIPE/HEAD ACETAMIN SPARES — “IF MORE THAN SURPRISED AFT